# Quantum-electrodynamical time-dependent density functional theory within Gaussian atomic basis - A tutorial for TDA-JC, RWA, Rabi, and PF

**Created by:** Khang Luong  
**Department of Chemistry, Brandeis University**  
#### **Reference:** https://doi.org/10.1063/5.0057542
---
## Overview
This tutorial examines the effect of Tamm-Dancoff approximation with Jaynes-Cummings (TDA-JC), rotating-wave approximation (RWA), Rabi, and Pauli-Fierz (PF) to model polariton spectrum of molecules in optical cavities.

This tutorial is divided to the follow parts: 

1. **Part 1 -** Configuration and TDA-JC Basics
2. **Part 2 -** Polariton spectrum as a function of coupling strength 
3. **Part 3 -** TDA-RWA application 
4. **Part 4 -** TDA-Rabi application
5. **Part 5 -** Capstone Exercise

---
## Environment Setup
### 1. (Optional) Create a Virtual Environment
It is recommended to use a virtual environment to avoid package conflicts. 

In, your terminal, run:

```bash
python3 -m venv venv
```
Next, activate the virtual environment with one of these commands.

MacOS:

```bash
source venv/bin/activate
```

Windows:

```bash
.venv\Scripts\activate
```

### 2. Install Dependencies
Install all required Python packages using:

```bash
pip install -r requirements.txt
```

### 3. Select the Jupyter Kernel
In the top-right corner of the notebook: 
1. Click **Kernel**
2. Select **Change Kernel**
3. Choose the `.venv` environment you just created

### Verify Installation
Run the following cell to import all required libraries and confirm that the environment is configured correctly.

### 📦 Imports & Configuration

In [ ]:
import numpy as np
from pyscf import gto, scf, tdscf
import scipy
import qed
#pip install qed later. 
import matplotlib.pyplot as plt
import warnings

warnings.filterwarnings("ignore")

# Color for printing
RED    = "\033[91m"
GREEN  = "\033[92m"
YELLOW = "\033[93m"
BLUE   = "\033[94m"
RESET  = "\033[0m"

#Unit Conversion:
HF_TO_EV = 27.2114
EV_TO_HF = 1 / HF_TO_EV

print("✅ All the imports are successful!")

--- 
# 🗂️ Part 1: Configuration
## 1.1 Molecular Structure and Ground State Calculation
We can specify the structure and basis of the molecule directly in the follow code block or any structural file. Then, we run the ground state SCF calculation:

In [ ]:
mol_file = 'ethene.xyz'  # Replace with your molecule file path
mol = gto.M(atom=mol_file, basis='6-311++G')
mf = scf.RKS(mol)
mf.xc = 'pbe0'
mf.kernel();

## 1.2 Theoretical Foundations

Under the Tamm-Dancoff approximation (TDA), the de-excitation block can be set to zero, resulting the TDA - PF (Tamm-Dancoff approximation Pauli-Fierz) model

$$
\begin{bmatrix}
\mathbf{A} + \Delta & \hbar \mathbf{g}^{\dagger} & \hbar \tilde{\mathbf{g}}^{\dagger} \\
\hbar \mathbf{g} & \hbar \mathbf{\omega} & 0 \\
\hbar \tilde{\mathbf{g}} & 0 & \hbar \mathbf{\omega}
\end{bmatrix}
\begin{bmatrix}
\mathbf{X} \\ \mathbf{M} \\ \mathbf{N}
\end{bmatrix}
=
\hbar\Omega^{\text{TDA-PF}}
\begin{bmatrix}
1 & 0 & 0 \\
0 & 1 & 0 \\
0 & 0 & -1 
\end{bmatrix}
\begin{bmatrix}
\mathbf{X} \\ \mathbf{M} \\ \mathbf{N}
\end{bmatrix}

\tag{1}
$$

where: 
+ $\mathbf{A}$ is the standard TDDFT electronic excitation matrix.
+ $\hbar\mathbf{\omega}$ is a diagonal matrix containing the cavity photon frequencies
+ $\mathbf{X}$ and $\mathbf{M}$ are the transition amplitudes for the electron and the dimensionless photon phase, respectively.
+ $\mathbf{g}$ is the electron-photon coupling strength.

If we further remove the dipole self-energy (DSE) $\Delta$, we can obtain the TDA-Rabi model

$$
\begin{bmatrix}
\mathbf{A} & \hbar \mathbf{g}^{\dagger} & \hbar \tilde{\mathbf{g}}^{\dagger} \\
\hbar \mathbf{g} & \hbar \mathbf{\omega} & 0 \\
\hbar \tilde{\mathbf{g}} & 0 & \hbar \mathbf{\omega}
\end{bmatrix}
\begin{bmatrix}
\mathbf{X} \\ \mathbf{M} \\ \mathbf{N}
\end{bmatrix}
=
\hbar\Omega^{\text{TDA-Rabi}}
\begin{bmatrix}
1 & 0 & 0 \\
0 & 1 & 0 \\
0 & 0 & -1 
\end{bmatrix}
\begin{bmatrix}
\mathbf{X} \\ \mathbf{M} \\ \mathbf{N}
\end{bmatrix}

\tag{2}
$$

If we neglect the counter-rotation terms (CRTs) $\hbar \tilde{\mathbf{g}}^{\dagger}$, we obtain the rotating wave approximation (RWA)

$$
\begin{bmatrix}
\mathbf{A} + \Delta & \hbar \mathbf{g}^{\dagger}\\
\hbar \mathbf{g} & \hbar \mathbf{\omega}
\end{bmatrix}
\begin{bmatrix}
\mathbf{X} \\ \mathbf{M}
\end{bmatrix}
=
\hbar\Omega^{\text{TDA-RWA}}
\begin{bmatrix}
1 & 0\\
0 & 1
\end{bmatrix}
\begin{bmatrix}
\mathbf{X} \\ \mathbf{M}
\end{bmatrix}

\tag{3}
$$

If both the DSE and CRT terms are negleted, we obtain the TDA-JC model 

$$
\begin{bmatrix}
\mathbf{A} & \hbar \mathbf{g}^{\dagger}\\
\hbar \mathbf{g} & \hbar \mathbf{\omega}
\end{bmatrix}
\begin{bmatrix}
\mathbf{X} \\ \mathbf{M}
\end{bmatrix}
=
\hbar\Omega^{\text{TDA-JC}}
\begin{bmatrix}
1 & 0\\
0 & 1
\end{bmatrix}
\begin{bmatrix}
\mathbf{X} \\ \mathbf{M}
\end{bmatrix}

\tag{4}
$$

### 💡 Physical Intuition behind the Models

While the equations above show the mathematical structure, it is important to understand the physical implications of choosing one model over another:

*   **JC & RWA (Few-Level Approximations):** These models are staples in quantum optics. They utilize the **Rotating Wave Approximation**, which simplifies the Hamiltonian by neglecting rapidly oscillating terms. These are computationally efficient but generally valid only in the **weak to strong coupling regimes** where the coupling strength $\lambda$ is much smaller than the transition frequency $\omega$.
*   **Rabi & PF (Ab Initio Models):** These are more physically complete. They include **Counter-Rotating Terms (CRTs)**, which become significant in the **ultra-strong coupling (USC)** regime. The Pauli-Fierz (PF) model is the most rigorous as it includes all terms derived from the minimal coupling Hamiltonian.
*   **The Dipole Self-Energy (DSE):** Represented by $\Delta$ in the PF and RWA equations, this term arises from the $A^2$ term in the Hamiltonian. It acts as a harmonic potential for the photon field shifted by the molecular dipole. Including DSE is crucial for maintaining the **lower bound of the energy spectrum**; without it, the ground state energy could theoretically become unstable as coupling strength increases.
*   **Counter-Rotating Terms (CRTs):** These terms allow for 'virtual' processes (e.g., simultaneous excitation of both the molecule and the cavity). Neglecting them (as in JC and RWA) is a common simplification, but it fails to capture the correct ground-state polarization and energy shifts when $\lambda$ is large.

## 1.3 Computational Workflow
### 1.3.1 Calculate Transition Dipole
1. We first need to construct the excited state solver on top of the mean-field object referece and obtain the excitation energies from the TDA eigenvalue equation.
2. Next, we compute the oscillator strengths, which measures how strongly light couples to a transition
$$
f_I = \frac{2}{3}\omega_I|\mu_{0I}|^2
$$
3. Then, we get the largest oscillator strength to compute the transition dipole moments, which is defined as
$$
\mathbf{\mu}_I^{\text{TDA-JC}}=\sum_{ai}X^{\text{TDA-JC}}_{I,ai}\mathbf{\mu}_{ai}
$$

**Note on Oscillator Strength:** The `oscillator_strength()` represents the dimensionless probability of a molecule interacting with electromagnetic radiation to undergo a specific electronic transition. A higher value indicates a brighter state. In cavity QED simulations, we typically align the cavity field polarization ($\vec{\epsilon}$) with the transition dipole moment ($\vec{d}_{ig}$) of the brightest state to achieve the maximum possible light-matter coupling strength.

In [ ]:
td = tdscf.TDA(mf)
td.nroots = 5 # Request 5 exicted states
td.kernel()

osc = td.oscillator_strength() # Compute oscillator strengths for each excitation
bright_idx = np.argmax(osc)
target_energy = td.e[bright_idx]

# Get transition dipole moment for the brightest excitation
trans_dip = td.transition_dipole()[bright_idx]
print(f"Targeting bright state: {bright_idx}, at {target_energy:.4f} a.u.")
print(f"Transition dipole moment (a.u.): {trans_dip}")

Next, we need to solve the full ab initio QED-TDDFT equations in the molecular orbital basis (Equation 4) 

### 1.3.2 Setup QED-TDA with JC Model

The TDA-JC method from ```qed``` package require a key which can contains:
| Category | Key | Type | Default | Description |
| --- | --- | --- | --- | --- |
| **Physical** | `cavity_freq` | float / array | **Required** | Photon mode energy in atomic units (Hartree). |
| | `cavity_mode` | array (3, N) | **Required** | Vector(s) defining coupling direction and vacuum field strength $\mathbf{\lambda}$. |
| | `cavity_model` | string | `'JC'` | Choice of Hamiltonian: `'JC'`, `'RWA'`, `'Rabi'`, or `'PF'`. |
| | `uniform_field` | boolean | `True` | Whether the vacuum field is spatially uniform across the molecule. |
| **Solver** | `nstates` | integer | `4` | Total number of **polaritonic** roots (eigenvalues) to solve for. |
| | `target_states` | string | `'polariton'` | Type of states to extract (e.g., `'polariton'`, `'exciton'`). |
| | `solver_algorithm`| string | `'davidson_qr'`| Algorithm: `'davidson_qr'` (ab initio) or `'direct'` (FewLevel). |
| | `tolerance` | float | `1e-8` | Convergence threshold for the eigenvalue solver residual. |
| | `max_cycle` | integer | `100` | Maximum iterations allowed for iterative solvers. |
| | `level_shift` | float | `0.0` | Numerical shift applied to diagonal elements to aid convergence. |
| **Tuning** | `resonance_state`| integer | `None` | Index of the TDA electronic state to which the photon should be tuned. |
| | `adjust_func` | string | `None` | Tuning strategy for frequency (e.g., `'average'`). Requires `resonance_state`.|
| **Collective**| `has_offdiag` | boolean | `False` | Include inter-fragment dipole-dipole interactions between molecules. |
| | `scale_coupling` | float | `0.0` | Scaling factor for coupling strength, typically $1/\sqrt{N_{frag}}$. |

In [ ]:
# Calculate unit vector of the transition dipole moment
unit_dip = trans_dip / np.linalg.norm(trans_dip)

#Define Cavity Parameters with lambda = 0.1 a.u. (can be change) and coupling direction along the transition dipole moment 
lambda_coupling = 0.1  # Coupling strength in atomic units
cavity_mode = (unit_dip * lambda_coupling).reshape(3, 1)  # Cavity mode vector along the transition dipole direction

key = {
    'cavity_mode': cavity_mode,
    'cavity_freq': np.array([target_energy]),
}

# Construct TDA-JC Model with qed package
cav_model = qed.JC(mf, key)
qed_td = qed.TDA(mf, td, cav_model, key)

# Run Calculation
qed_td.nroots = 8
qed_td.kernel();

print(f"\n{GREEN}Polaritonic states computed successfully!{RESET}")
print(f"Polaritonic energies (in eV): {qed_td.e * HF_TO_EV}")
print(f"Transition dipole moments (a.u.): {qed_td.transition_dipole()}")

### 1.3.2 🏋️ **Exercise: The Effect of Detuning**
Here, we tuned the cavity frequency exactly to the electronic transition ($\delta = 0$).
*   **Task:** In Part 1.3.2, manually set `cavity_freq` to be 0.5 eV higher than the `target_energy`.
*   **Question:** Look at the resulting polariton energies and the photon contribution. How does "detuning" the cavity affect the mixing between the exciton and the photon? Does one state become "more photonic" than the other?

---
# 🔬 Part 2: Scanning Coupling Strength ($\lambda$)
Before we scan the coupling strength, it is crucial to define our parameters physically. The fundamental coupling strength ($\lambda$) is expressed in atomic units ($e^{-1} a_0^{-1/2}$). 

Physically, increasing $\lambda$ corresponds to a tighter confinement of the electromagnetic field—essentially decreasing the effective mode volume of the optical cavity. This tighter spatial confinement leads to a stronger, more entangled interaction between the vacuum photon field and the molecule's transition dipole moment. 

To ensure maximum resonance, we will dynamically align the cavity polarization vector to perfectly match the molecule's transition dipole unit vector at every step.

In [ ]:
lambdas = np.linspace(0.0 , 0.10, 12)
cavity_freqs = np.array([target_energy])

JC_energies = []
JC_photon_contribution = [] 

for lam in lambdas:
    print(f"{GREEN}Coupling lam = {lam:.3f} au...{RESET}", end="\r")
    # 1. Update mode adn re-run TDA-JC
    unit_dip = trans_dip / np.linalg.norm(trans_dip)
    cavity_mode = (unit_dip * lam).reshape(3, 1)
    key = {'cavity_mode': cavity_mode, 'cavity_freq': cavity_freqs}
    cav_jc = qed.JC(mf, key)
    td_jc = qed.TDA(mf, td, cav_jc, key)
    td_jc.nroots = 5
    td_jc.kernel()
    print(f"Coupling: {lam:.3f} a.u., Polariton Energies: {td_jc.e}")
    # 2. Extract energies
    JC_energies.append(td_jc.e)

    # 3. Extract photon contribution
    p = cav_jc.get_mns_weight(td_jc.mn)
    JC_photon_contribution.append(p.flatten())
    print(f"{YELLOW}----------------------------------------------------------------{RESET}")

JC_energies = np.array(JC_energies)
JC_photon_contribution = np.array(JC_photon_contribution)

## 💡 Generate Plot

+ Since the cavity frequency is tuned excactly to the brightest electronic transition, the photon mode and the electronic state are **degenerate** at $\lambda = 0$
+ When $\lambda > 0$, this degenerate pair splits into the Lower Polariton (Root 0) and Upper Polariton (Root 1)
+ Root 2 is the next electronic state

In [ ]:
fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(12, 6))

# Energy Plot
ax1.plot(lambdas, JC_energies[:, 0] * HF_TO_EV, 'ro-', markersize=6, alpha=0.7, label='Lower Polariton (LP)')
ax1.plot(lambdas, JC_energies[:, 1] * HF_TO_EV, 'bo-', markersize=6, alpha=0.7, label='Upper Polariton (UP)')
ax1.plot(lambdas, JC_energies[:, 2] * HF_TO_EV, 'go-', markersize=6, alpha=0.7, label='Third Eigenstate')
ax1.axhline(target_energy * HF_TO_EV, color='gray', linestyle='--', label='Resonant Frequency')
ax1.axvline(0, color='black', linewidth=1, linestyle='-', alpha=0.5)

# Focus on the splitting area
y_min = np.min(JC_energies[:, :2] * HF_TO_EV) - 0.5
y_max = np.max(JC_energies[:, :2] * HF_TO_EV) + 1.0
ax1.set_ylim(y_min, y_max)

ax1.set_xlabel('Coupling Strength $\lambda$ (a.u.)')
ax1.set_ylabel('Energy (eV)')
ax1.set_title('Polariton Energies vs Coupling')
ax1.legend()
ax1.grid(alpha=0.3)

# Photon Contribution Plot
mask = lambdas >= 1e-8
ax2.plot(lambdas[mask], JC_photon_contribution[:, 0][mask], 'ro-', markersize=6, alpha=0.7, label='LP Photon Character')
ax2.plot(lambdas[mask], JC_photon_contribution[:, 1][mask], 'bo-', markersize=6, alpha=0.7, label='UP Photon Character')
ax2.axhline(0.5, color='black', linestyle=':', alpha=0.6, label='50% Photon/Exciton')
ax2.axvline(0, color='black', linewidth=1, linestyle='-', alpha=0.5)

ax2.set_xlabel('Coupling Strength $\lambda$ (a.u.)')
ax2.set_ylabel('Photon Contribution')
ax2.set_title('Hybrid Nature of Polaritons')
ax2.legend()
ax2.grid(alpha=0.3)

fig.tight_layout()
plt.show()

## 🏋️ Excercise: Exploring a New Molecule (Formaldehyde)
Apply the tutorial workflow to a different system.
*   **Task:** Replace the `ethene.xyz` geometry with Formaldehyde. You can define it directly in the code:
    ```python
    mol_geometry = """
    C   0.0000000   0.0000000   0.0000000
    O   0.0000000   0.0000000   1.2200000
    H   0.0000000   0.9400000  -0.5800000
    H   0.0000000  -0.9400000  -0.5800000
    """
    mol = gto.M(atom=mol_geometry, basis='6-311++G')
    ```
*   **Question:** Identify the brightest state for Formaldehyde. Is it at a higher or lower energy than Ethene?

---
# Part 3: Tamm-Dancoff with Rotating Wave Approximation (TDA-RWA)

## 3.1 Computational Workflow
Equation 3 is what we need to solve for this method. As such, we can follow the same structure as TDA-JC, but replace the JC attribute with RWA.

In [ ]:
cav_rwa = qed.RWA(mf, key)
qed_rwa = qed.TDA(mf, td, cav_rwa, key)

qed_rwa.nroots = 8
qed_rwa.kernel()

## 3.2 Compare TDA-JC vs TDA-RWA

In [ ]:
RWA_energies = []
RWA_photon_contribution = []

for lam in lambdas:
    print(f"{GREEN}Coupling lam = {lam:.3f} au...{RESET}", end="\r")
    key['cavity_mode'] = (unit_dip * lam).reshape(3, 1)
    cav_rwa = qed.RWA(mf, key)
    td_rwa = qed.TDA(mf, td, cav_rwa, key)
    td_rwa.nroots = 5
    td_rwa.kernel()
    print(f"Coupling: {lam:.3f} a.u., Polariton Energies: {td_rwa.e}")
    p = cav_rwa.get_mns_weight(td_rwa.mn)
    RWA_photon_contribution.append(p.flatten())
    RWA_energies.append(td_rwa.e)
    print(f"{YELLOW}----------------------------------------------------------------{RESET}")
    
RWA_energies = np.array(RWA_energies)
RWA_photon_contribution = np.array(RWA_photon_contribution)

fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(10, 6))

ax1.plot(lambdas, JC_energies[:, 0] * HF_TO_EV, color='tab:red', marker='o', linestyle='-', markersize=6, alpha=0.7, label='JC Lower Polariton')
ax1.plot(lambdas, JC_energies[:, 1] * HF_TO_EV, color='tab:blue', marker='o', linestyle='-', markersize=6, alpha=0.7, label='JC Upper Polariton')
ax1.plot(lambdas, JC_energies[:, 2] * HF_TO_EV, color='tab:green', marker='o', linestyle='-', markersize=6, alpha=0.7, label='JC Third Eigenstate')
ax1.plot(lambdas, RWA_energies[:, 0] * HF_TO_EV, color='tab:orange', marker='s', linestyle='--', markersize=6, alpha=0.7, label='RWA Lower Polariton')
ax1.plot(lambdas, RWA_energies[:, 1] * HF_TO_EV, color='tab:purple', marker='s', linestyle='--', markersize=6, alpha=0.7, label='RWA Upper Polariton')
ax1.plot(lambdas, RWA_energies[:, 2] * HF_TO_EV, color='tab:brown', marker='s', linestyle='--', markersize=6, alpha=0.7, label='RWA Third Eigenstate')
ax1.axhline(target_energy * HF_TO_EV, color='gray', linestyle='--', label='Underlying Bright State')

ax1.set_xlabel('Coupling Strength (a.u.)')
ax1.set_ylabel('Energy (eV)')
ax1.set_title('Polariton Energies vs Coupling Strength')
ax1.legend()
ax1.grid(alpha=0.3)

mask = lambdas >= 1e-8

ax2.plot(lambdas[mask], JC_photon_contribution[:, 0][mask], color='tab:red', marker='o', linestyle='-', markersize=6, alpha=0.7, label='JC Lower Polariton')
ax2.plot(lambdas[mask], JC_photon_contribution[:, 1][mask], color='tab:blue', marker='o', linestyle='-', markersize=6, alpha=0.7, label='JC Upper Polariton')
ax2.plot(lambdas[mask], JC_photon_contribution[:, 2][mask], color='tab:green', marker='o', linestyle='-', markersize=6, alpha=0.7, label='JC Third Polariton')
ax2.plot(lambdas[mask], RWA_photon_contribution[:, 0][mask], color='tab:orange', marker='s', linestyle='--', markersize=6, alpha=0.7, label='RWA Lower Polariton')
ax2.plot(lambdas[mask], RWA_photon_contribution[:, 1][mask], color='tab:purple', marker='s', linestyle='--', markersize=6, alpha=0.7, label='RWA Upper Polariton')
ax2.plot(lambdas[mask], RWA_photon_contribution[:, 2][mask], color='tab:brown', marker='s', linestyle='--', markersize=6, alpha=0.7, label='RWA Third Polariton')

ax2.set_xlabel('Coupling Strength (a.u.)')
ax2.set_ylabel('Photon Contribution')
ax2.set_title('Photon Contribution vs Coupling Strength')
ax2.legend()
ax2.grid(alpha=0.3)

fig.tight_layout()
plt.show()

---
# Part 4: Tamm-Dancoff approximation with Rabi model
## 4.1 Computational Workflow
The key equation for this is Equation 2. To implement this, we follow the same procedure as before:

In [ ]:
cav_rabi = qed.Rabi(mf, key) # Using Rabi model
qed_rabi = qed.TDA(mf, td, cav_rabi, key)

qed_rabi.nroots = 8
qed_rabi.kernel()

## 4.2 Compare TDA-JC, TDA-RWA, and TDA-Rabi

**Important note:** JC/RWA uses a **single** photon amplitude $M$ so that the photon wieght is simply $M^2$. However, Rabi/PF uses **two** photon amplitude: $M$ (photon annihilation) and $N$ (photon creation). The actual "photon number" is there fore defined as $M^2 - N^2$

In [ ]:
Rabi_energies = []
Rabi_photon_contribution = []

for lam in lambdas:
    print(f"{BLUE}Coupling lam = {lam:.3f}, au...{RESET}", end="\r")
    key['cavity_mode'] = (unit_dip * lam).reshape(3, 1)
    cav_rabi = qed.Rabi(mf, key)
    td_rabi = qed.TDA(mf, td, cav_rabi, key)
    td_rabi.nroots = 5
    td_rabi.kernel()
    print(f"Coupling: {lam:.3f} a.u., Polariton Energies: {td_rabi.e}")
    p = cav_rabi.get_mns_weight(td_rabi.mn)
    Rabi_photon_contribution.append(p)
    Rabi_energies.append(td_rabi.e)
    print(f"{YELLOW}----------------------------------------------------------------{RESET}")

Rabi_energies = np.array(Rabi_energies)
Rabi_photon_contribution = np.array(Rabi_photon_contribution)

fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(10, 6))

state_colors = ['tab:red', 'tab:blue', 'tab:green']

# JC
for i, label in enumerate([
    'Lower Polariton',
    'Upper Polariton',
    'Third Eigenstate'
]):
    ax1.plot(
        lambdas,
        JC_energies[:, i] * HF_TO_EV, color=state_colors[i], marker='o', linestyle='-',
        linewidth=2,
        markersize=6,
        alpha=0.85,
        label=f'JC {label}'
    )

# RWA
for i, label in enumerate([
    'Lower Polariton',
    'Upper Polariton',
    'Third Eigenstate'
]):
    ax1.plot(
        lambdas,
        RWA_energies[:, i] * HF_TO_EV, color=state_colors[i], marker='s', linestyle='--',
        linewidth=2,
        markersize=6,
        alpha=0.85,
        label=f'RWA {label}'
    )

# Rabi
for i, label in enumerate([
    'Lower Polariton',
    'Upper Polariton',
    'Third Eigenstate'
]):
    ax1.plot(
        lambdas,
        Rabi_energies[:, i] * HF_TO_EV, color=state_colors[i], marker='^', linestyle=':',
        linewidth=2.5,
        markersize=7,
        alpha=0.9,
        label=f'Rabi {label}'
    )

ax1.axhline(
    target_energy * HF_TO_EV, color='black', linestyle='-.',
    linewidth=2,
    alpha=0.7,
    label='Underlying Bright State'
)

ax1.set_xlabel('Coupling Strength (a.u.)')
ax1.set_ylabel('Energy (eV)')
ax1.set_title('Polariton Energies vs Coupling Strength')
ax1.legend(ncol=3, fontsize=9)
ax1.grid(alpha=0.3)


mask = lambdas >= 1e-8

# JC
for i, label in enumerate([
    'Lower Polariton',
    'Upper Polariton',
    'Third Polariton'
]):
    ax2.plot(
        lambdas[mask],
        JC_photon_contribution[:, i][mask], color=state_colors[i], marker='o', linestyle='-',
        linewidth=2,
        markersize=6,
        alpha=0.85,
        label=f'JC {label}'
    )

# RWA
for i, label in enumerate([
    'Lower Polariton',
    'Upper Polariton',
    'Third Polariton'
]):
    ax2.plot(
        lambdas[mask],
        RWA_photon_contribution[:, i][mask], color=state_colors[i], marker='s', linestyle='--',
        linewidth=2,
        markersize=6,
        alpha=0.85,
        label=f'RWA {label}'
    )

# Rabi
for i, label in enumerate([
    'Lower Polariton',
    'Upper Polariton',
    'Third Polariton'
]):
    ax2.plot(
        lambdas[mask],
        Rabi_photon_contribution[:, i][mask], color=state_colors[i], marker='^', linestyle=':',
        linewidth=2.5,
        markersize=7,
        alpha=0.9,
        label=f'Rabi {label}'
    )

ax2.set_xlabel('Coupling Strength (a.u.)')
ax2.set_ylabel('Photon Contribution')
ax2.set_title('Photon Character vs Coupling Strength')
ax2.legend(ncol=3, fontsize=9)
ax2.grid(alpha=0.3)

fig.tight_layout()
plt.show()

# ➡️ **Summary Table**

| **Model** | **DSE ($\Delta$)** | **CRTs** | **Complexity** | **Use case** |
| --- | --- | --- | --- | --- |
| JC | No | No | Lowest | Weak coupling, theoretical simplicity | 
| RWA | Yes | No | Low | Moderate coupling |
| Rabi | No | Yes | Medium | Strong coupling without DSE |
| PF | Yes | Yes | Highest | Ultra-strong coupling, ab initio accuracy |


---
## 🏆 Part 5: Capstone Challenge – The Ultra-Strong Breakdown

Throughout this lesson, you have run individual calculations for the Jaynes-Cummings (JC), Rotating-Wave Approximation (RWA), and Rabi models. Now, we will visualize exactly why simple quantum optical models break down when light-matter interactions become extreme.

### The Task: 
Write a script that calculates the energy of the **Lower Polariton** using these three cavity models across a wide range of coupling strengths.

**1. Setup the Scan:**

Initialize Ethene and target its brightest transition, just as we did in Part 1. Set your coupling array to reach deep into the ultra-strong regime: `lambdas = np.linspace(0.0, 0.20, 20)`.

**2. The 3-Model Loop:**

Inside your loop over `lambdas`, dynamically align the cavity to the transition dipole. Then, initialize and solve all three QED-TDA models:
* `qed.JC`
* `qed.RWA`
* `qed.Rabi`

Store the lowest polariton energy for each model at every step.

**3. Visualizing the Breakdown:**

Plot all three arrays on the same graph: `Coupling Strength (a.u.)` on the x-axis, and `Lower Polariton Energy (eV)` on the y-axis. 

### Analysis Questions:
* **The Symmetry Breaking:** At weak coupling ($\lambda < 0.05$), do the models agree? 
* **The Ground State Collapse:** Look at the JC and Rabi models (which both lack the Dipole Self-Energy term). What happens to their predicted energies as $\lambda$ approaches 0.20? 
* **The RWA Fix:** How does the RWA model behave differently at high coupling, and why? 

*(Note: While RWA fixes the catastrophic collapse, it is still an incomplete picture. In Lesson 2, we will introduce the full Pauli-Fierz Hamiltonian to capture the missing physics!)*